

Verifies the predefined train/valid/test split and the one-image-one-horse assumption.



## 1. Imports and paths

In [1]:
from pathlib import Path
import os, json, shutil, zipfile, hashlib, re, warnings
from datetime import datetime, timezone
import pandas as pd
import numpy as np


BASE_DIR = Path("/content")
PROJECT_NAME = "project_thermography_equine"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME

DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
MODEL_SELECTION_DIR = OUTPUT_ROOT / "model_selection"

for p in [PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR, ANNOTATIONS_DIR,
          OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, MODEL_SELECTION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project paths initialized")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPLIT_DATA_DIR:", SPLIT_DATA_DIR)
print("ANNOTATIONS_DIR:", ANNOTATIONS_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Project paths initialized
PROJECT_ROOT: /content/project_thermography_equine
SPLIT_DATA_DIR: /content/project_thermography_equine/data/dataset_split
ANNOTATIONS_DIR: /content/project_thermography_equine/data/annotations
OUTPUT_ROOT: /content/project_thermography_equine/outputs


## 2. Load finalized metadata

In [2]:
metadata_candidates = [
    # Prefer the QC-filtered analysis dataset produced by notebook 03.
    # This prevents excluded/low-quality images from entering split/leakage summaries and downstream manifests.
    CONFIG_DIR / 'master_metadata_qc.csv',
    METADATA_DIR / 'master_metadata_qc.csv',
    REPORTS_DIR / 'master_metadata_qc.csv',
    CONFIG_DIR / 'master_metadata_with_annotations.csv',
    CONFIG_DIR / 'master_metadata.csv',
    METADATA_DIR / 'master_metadata_with_annotations.csv',
    METADATA_DIR / 'master_metadata.csv',
]
metadata_path = next((p for p in metadata_candidates if p.exists()), None)
if metadata_path is None:
    raise FileNotFoundError('No master metadata file found. Run notebooks 01, 02 and 03 first.')

master_df = pd.read_csv(metadata_path)
if 'included_in_final_analysis' in master_df.columns:
    before_qc_filter = len(master_df)
    master_df = master_df[master_df['included_in_final_analysis'].astype(bool)].copy()
    print(f'Applied QC inclusion filter: {len(master_df)} / {before_qc_filter} records retained for split/leakage checks.')
else:
    print('WARNING: included_in_final_analysis column not found; split/leakage checks will use all metadata rows.')
with open(CONFIG_DIR / 'analysis_config.json', 'r', encoding='utf-8') as f:
    analysis_config = json.load(f)
with open(CONFIG_DIR / 'study_protocol.json', 'r', encoding='utf-8') as f:
    study_protocol = json.load(f)

print('Loaded:', metadata_path)
print('Rows:', len(master_df))
display(master_df.head())

Applied QC inclusion filter: 347 / 347 records retained for split/leakage checks.
Loaded: /content/project_thermography_equine/outputs/config/master_metadata_qc.csv
Rows: 347


,horse_id,image_id,image_name,image_stem,image_ext,split,folder_label,label_clinical,label_binary,image_path,...,width,height,mean_intensity,std_intensity,laplacian_variance,automated_exclusion,manual_qc_decision,manual_qc_reason,included_after_qc,included_in_final_analysis
0,0A0F5,0A0F5,0A0F5.jpg,0A0F5,.jpg,test,healthy,healthy,0,/content/project_thermography_equine/data/data...,...,492,331,25.173998,56.775894,121.968100,False,include,NaN,True,True
1,1CGFM,1CGFM,1CGFM.jpg,1CGFM,.jpg,test,healthy,healthy,0,/content/project_thermography_equine/data/data...,...,492,331,40.948402,51.187489,133.616760,False,include,NaN,True,True
2,1RDFC,1RDFC,1RDFC.jpg,1RDFC,.jpg,test,healthy,healthy,0,/content/project_thermography_equine/data/data...,...,492,331,21.600779,48.384766,58.460165,False,include,NaN,True,True
3,31W0A,31W0A,31W0A.jpg,31W0A,.jpg,test,healthy,healthy,0,/content/project_thermography_equine/data/data...,...,492,331,59.057777,59.399006,165.383526,False,include,NaN,True,True
4,6BGWZ,6BGWZ,6BGWZ.jpg,6BGWZ,.jpg,test,healthy,healthy,0,/content/project_thermography_equine/data/data...,...,492,331,25.508131,53.596935,106.242751,False,include,NaN,True,True


## 3. Split composition

In [3]:
split_summary = master_df.groupby(['split','label_clinical','label_binary']).size().reset_index(name='n_images')
split_totals = master_df.groupby('split').size().reset_index(name='n_images')
class_totals = master_df.groupby(['label_clinical','label_binary']).size().reset_index(name='n_images')

display(split_summary)
display(split_totals)
display(class_totals)


for split, grp in master_df.groupby('split'):
    classes = set(grp['label_clinical'])
    if len(classes) < 2:
        print(f'WARNING: split {split} contains only one class: {classes}')

,split,label_clinical,label_binary,n_images
0,test,healthy,0,40
1,test,pathological,1,13
2,train,healthy,0,179
3,train,pathological,1,63
4,valid,healthy,0,38
5,valid,pathological,1,14


,split,n_images
0,test,53
1,train,242
2,valid,52


,label_clinical,label_binary,n_images
0,healthy,0,257
1,pathological,1,90


## 4. Leakage checks

In [4]:
leakage_rows = []

def add_leakage_check(name, status, n_problematic, details=''):
    leakage_rows.append({'check': name, 'status': bool(status), 'n_problematic': int(n_problematic), 'details': str(details)})

# one image = one horse
horse_split_counts = master_df.groupby('horse_id')['split'].nunique().reset_index(name='n_splits')
horse_cross_split = horse_split_counts[horse_split_counts['n_splits'] > 1]
add_leakage_check('no_horse_id_cross_split_overlap', horse_cross_split.empty, len(horse_cross_split), horse_cross_split.head(20).to_dict('records'))

horse_counts = master_df['horse_id'].value_counts()
duplicate_horse_ids = horse_counts[horse_counts > 1]
add_leakage_check('one_image_per_horse_id', duplicate_horse_ids.empty, len(duplicate_horse_ids), duplicate_horse_ids.head(20).to_dict())

# image names
name_split_counts = master_df.groupby('image_name')['split'].nunique().reset_index(name='n_splits')
name_cross_split = name_split_counts[name_split_counts['n_splits'] > 1]
add_leakage_check('no_image_name_cross_split_overlap', name_cross_split.empty, len(name_cross_split), name_cross_split.head(20).to_dict('records'))

name_counts = master_df['image_name'].value_counts()
duplicate_names = name_counts[name_counts > 1]
add_leakage_check('no_duplicate_image_names', duplicate_names.empty, len(duplicate_names), duplicate_names.head(20).to_dict())

# file hashes if available
if 'file_sha256' in master_df.columns:
    hash_split_counts = master_df.groupby('file_sha256')['split'].nunique().reset_index(name='n_splits')
    hash_cross_split = hash_split_counts[hash_split_counts['n_splits'] > 1]
    add_leakage_check('no_file_hash_cross_split_overlap', hash_cross_split.empty, len(hash_cross_split), hash_cross_split.head(20).to_dict('records'))

    hash_counts = master_df['file_sha256'].value_counts()
    duplicate_hashes = hash_counts[hash_counts > 1]
    add_leakage_check('no_duplicate_file_hashes', duplicate_hashes.empty, len(duplicate_hashes), duplicate_hashes.head(20).to_dict())

leakage_summary = pd.DataFrame(leakage_rows)
display(leakage_summary)

if not leakage_summary['status'].all():
    print('WARNING: Leakage checks detected problems. Review detailed reports before modeling.')
else:
    print('No split leakage detected based on available identifiers and hashes.')

,check,status,n_problematic,details
0,no_horse_id_cross_split_overlap,True,0,[]
1,one_image_per_horse_id,True,0,{}
2,no_image_name_cross_split_overlap,True,0,[]
3,no_duplicate_image_names,True,0,{}
4,no_file_hash_cross_split_overlap,True,0,[]
5,no_duplicate_file_hashes,True,0,{}


No split leakage detected based on available identifiers and hashes.


## 5. Annotation conflict status

In [5]:
if 'annotation_label_conflict' in master_df.columns:
    conflict_df = master_df[master_df['annotation_label_conflict'].astype(bool)].copy()
    print('Annotation/clinical label conflicts:', len(conflict_df))
    if len(conflict_df):
        display(conflict_df[['split','image_name','label_clinical','folder_label','has_hotspot_expert1','has_hotspot_expert2','relative_image_path']])
else:
    conflict_df = pd.DataFrame()
    print('No annotation conflict column available. Run notebook 02 to add annotation flags.')

Annotation/clinical label conflicts: 0


## 6. Save manifests and reports

In [6]:
split_manifest = master_df.sort_values(['split','label_clinical','image_name']).reset_index(drop=True)
split_manifest.to_csv(REPORTS_DIR / 'split_manifest.csv', index=False)
split_manifest.to_csv(CONFIG_DIR / 'split_manifest.csv', index=False)

for split in analysis_config['splits']:
    split_df = split_manifest[split_manifest['split'] == split]
    split_df.to_csv(REPORTS_DIR / f'{split}_manifest.csv', index=False)
    split_df[['horse_id','image_name','label_clinical','label_binary','relative_image_path']].to_csv(REPORTS_DIR / f'{split}_ids.csv', index=False)

split_summary.to_csv(REPORTS_DIR / 'split_summary.csv', index=False)
split_totals.to_csv(REPORTS_DIR / 'split_summary_by_split.csv', index=False)
class_totals.to_csv(REPORTS_DIR / 'split_summary_by_class.csv', index=False)
leakage_summary.to_csv(REPORTS_DIR / 'leakage_summary.csv', index=False)
if 'conflict_df' in globals():
    conflict_df.to_csv(REPORTS_DIR / 'split_annotation_label_conflicts.csv', index=False)

# Detailed duplicates
if len(horse_cross_split):
    master_df[master_df['horse_id'].isin(horse_cross_split['horse_id'])].to_csv(REPORTS_DIR / 'leakage_cross_split_horse_ids.csv', index=False)
if len(name_cross_split):
    master_df[master_df['image_name'].isin(name_cross_split['image_name'])].to_csv(REPORTS_DIR / 'leakage_cross_split_image_names.csv', index=False)
if 'file_sha256' in master_df.columns and len(hash_cross_split):
    master_df[master_df['file_sha256'].isin(hash_cross_split['file_sha256'])].to_csv(REPORTS_DIR / 'leakage_cross_split_file_hashes.csv', index=False)

print('Saved split and leakage reports.')

Saved split and leakage reports.


## 7. Final readiness for modeling

In [7]:
modeling_readiness = pd.DataFrame([
    {'criterion': 'train_valid_test_present', 'status': set(analysis_config['splits']).issubset(set(master_df['split'])), 'details': sorted(master_df['split'].unique())},
    {'criterion': 'both_classes_in_train', 'status': master_df[master_df['split']=='train']['label_clinical'].nunique() == 2, 'details': master_df[master_df['split']=='train']['label_clinical'].value_counts().to_dict()},
    {'criterion': 'both_classes_in_valid', 'status': master_df[master_df['split']=='valid']['label_clinical'].nunique() == 2, 'details': master_df[master_df['split']=='valid']['label_clinical'].value_counts().to_dict()},
    {'criterion': 'both_classes_in_test', 'status': master_df[master_df['split']=='test']['label_clinical'].nunique() == 2, 'details': master_df[master_df['split']=='test']['label_clinical'].value_counts().to_dict()},
    {'criterion': 'no_split_leakage', 'status': leakage_summary['status'].all(), 'details': leakage_summary.to_dict('records')},
    {'criterion': 'no_annotation_label_conflicts', 'status': len(conflict_df) == 0, 'details': f'{len(conflict_df)} conflicts'}
])
modeling_readiness.to_csv(REPORTS_DIR / 'modeling_readiness_report.csv', index=False)
modeling_readiness.to_csv(CONFIG_DIR / 'modeling_readiness_report.csv', index=False)
display(modeling_readiness)

if not modeling_readiness['status'].all():
    print('CAUTION: Data are usable for code development, but final publication analysis requires review of failed readiness criteria.')
else:
    print('Ready for downstream modeling notebooks.')

,criterion,status,details
0,train_valid_test_present,True,"[test, train, valid]"
1,both_classes_in_train,True,"{'healthy': 179, 'pathological': 63}"
2,both_classes_in_valid,True,"{'healthy': 38, 'pathological': 14}"
3,both_classes_in_test,True,"{'healthy': 40, 'pathological': 13}"
4,no_split_leakage,True,"[{'check': 'no_horse_id_cross_split_overlap', ..."
5,no_annotation_label_conflicts,True,0 conflicts


Ready for downstream modeling notebooks.
